In [ ]:
import os
import pickle
import numpy as np
import plotly.graph_objects as go
from sklearn.decomposition import PCA
from tqdm.notebook import tqdm

# ==========================================
# 1. Configuration
# ==========================================
DATA_DIR = "/Volumes/T9/Concepts/TCAV_neural/fg/concepts/alpha_O2_high/1_concept_activations"
ANTI_CONCEPT_DIR = "/Volumes/T9/Concepts/TCAV_neural/fg/concepts/alpha_O2_low/1_concept_activations"
GRAD_DIR = "/Volumes/T9/Concepts/TCAV_neural/fg/target_gradients" 
CAV_DIR = "/Volumes/T9/Concepts/TCAV_neural/fg/concepts/alpha_O2_high/2_real_cavs" # <--- ADDED

LAYERS = [0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11]
NUM_RANDOM_RUNS = 100
MAX_RANDOM_SAMPLES = 10000 

# ==========================================
# 2. Helper Functions
# ==========================================
def load_pkl(filepath):
    if os.path.exists(filepath):
        with open(filepath, 'rb') as f:
            return pickle.load(f)
    return None

def load_layer_data(layer_id, data_dir, anti_concept_dir, num_random_runs, max_random):
    features_list, labels_list = [], []

    target_data = load_pkl(os.path.join(data_dir, f"target_class_set_layer_{layer_id}.pkl"))
    if target_data is not None:
        features_list.append(target_data)
        labels_list.extend(['open screen'] * len(target_data))

    neg_target_data = load_pkl(os.path.join(data_dir, f"negative_target_class_set_layer_{layer_id}.pkl"))
    if neg_target_data is not None:
        features_list.append(neg_target_data)
        labels_list.extend(['closed screen'] * len(neg_target_data))

    concept_data = load_pkl(os.path.join(data_dir, f"concept_set_layer_{layer_id}.pkl"))
    if concept_data is not None:
        features_list.append(concept_data)
        labels_list.extend(['Alpha_O2_high'] * len(concept_data))

    anti_concept_data = load_pkl(os.path.join(anti_concept_dir, f"concept_set_layer_{layer_id}.pkl"))
    if anti_concept_data is not None:
        features_list.append(anti_concept_data)
        labels_list.extend(['Alpha_O2_low'] * len(anti_concept_data))

    random_features = []
    for i in range(num_random_runs):
        rnd_data = load_pkl(os.path.join(data_dir, f"random_run_{i}_layer_{layer_id}.pkl"))
        if rnd_data is not None: random_features.append(rnd_data)
    
    if random_features:
        all_random = np.vstack(random_features)
        if len(all_random) > max_random:
            indices = np.random.choice(len(all_random), max_random, replace=False)
            all_random = all_random[indices]
        features_list.append(all_random)
        labels_list.extend(['Random (Null)'] * len(all_random))

    if not features_list: return None, None
    return np.vstack(features_list), np.array(labels_list)

# ==========================================
# 3. Visualization Function
# ==========================================
def plot_layer(layer_id):
    print(f"Loading data for Layer {layer_id}...")
    X, y = load_layer_data(layer_id, DATA_DIR, ANTI_CONCEPT_DIR, NUM_RANDOM_RUNS, MAX_RANDOM_SAMPLES)
    if X is None: return

    pca = PCA(n_components=3)
    X_pca = pca.fit_transform(X)
    var_explained = pca.explained_variance_ratio_ * 100
    
    fig = go.Figure()

    styles = {
        'Random (Null)': dict(color='lightgrey', size=5, opacity=0.3, symbol='circle'),
        'closed screen': dict(color='orange', size=5, opacity=0.3, symbol='circle'),
        'open screen': dict(color='blue', size=5, opacity=0.3, symbol='circle'),
        'Alpha_O2_low': dict(color='red', size=8, opacity=0.8, symbol='diamond'),
        'Alpha_O2_high': dict(color='lime', size=8, opacity=0.8, symbol='diamond')
    }

    # Plot Scatter Data
    for label in ['Random (Null)', 'closed screen', 'open screen', 'Alpha_O2_low', 'Alpha_O2_high']:
        idx = np.where(y == label)[0]
        if len(idx) == 0: continue
        fig.add_trace(go.Scatter3d(
            x=X_pca[idx, 0], y=X_pca[idx, 1], z=X_pca[idx, 2],
            mode='markers', marker=styles[label], name=f"{label} (n={len(idx)})"
        ))

    # --- 1. TARGET GRADIENT ARROW (Black) ---
    grad_data = load_pkl(os.path.join(GRAD_DIR, f"target_gradients_layer_{layer_id}.pkl"))
    if grad_data is not None:
        mean_grad = np.mean(grad_data, axis=0) 
        proj_grad = np.dot(mean_grad, pca.components_.T)
        
        idx_neg = np.where(y == 'closed screen')[0]
        idx_pos = np.where(y == 'open screen')[0]
        
        if len(idx_neg) > 0 and len(idx_pos) > 0:
            c_neg = X_pca[idx_neg].mean(axis=0)
            c_pos = X_pca[idx_pos].mean(axis=0)
            
            v_norm = proj_grad / np.linalg.norm(proj_grad)
            scale = np.linalg.norm(c_pos - c_neg) 
            end_pt = c_neg + (v_norm * scale)
            
            fig.add_trace(go.Scatter3d(
                x=[c_neg[0], end_pt[0]], y=[c_neg[1], end_pt[1]], z=[c_neg[2], end_pt[2]],
                mode='lines', line=dict(color='black', width=6),
                name="Target Gradient"
            ))
            fig.add_trace(go.Cone(
                x=[end_pt[0]], y=[end_pt[1]], z=[end_pt[2]],
                u=[v_norm[0]], v=[v_norm[1]], w=[v_norm[2]],
                sizemode="absolute", sizeref=scale*0.15, anchor="tail",
                colorscale=[[0, 'black'], [1, 'black']], showscale=False,
                name="Gradient Head"
            ))

    # --- 2. FILTER CAV ARROW (Green) ---
    cav_data = load_pkl(os.path.join(CAV_DIR, f"cavs_concept_set_layer_{layer_id}.pkl"))
    if cav_data is not None and 'filter' in cav_data:
        # Get the mean Filter CAV across all 100 runs
        mean_cav = np.mean(cav_data['filter'], axis=0)
        proj_cav = np.dot(mean_cav, pca.components_.T)
        
        idx_rnd = np.where(y == 'Random (Null)')[0]
        idx_concept = np.where(y == 'Alpha_O2_high')[0]
        
        if len(idx_rnd) > 0 and len(idx_concept) > 0:
            c_rnd = X_pca[idx_rnd].mean(axis=0)
            c_concept = X_pca[idx_concept].mean(axis=0)
            
            v_norm_cav = proj_cav / np.linalg.norm(proj_cav)
            scale_cav = np.linalg.norm(c_concept - c_rnd)
            end_pt_cav = c_rnd + (v_norm_cav * scale_cav)
            
            fig.add_trace(go.Scatter3d(
                x=[c_rnd[0], end_pt_cav[0]], y=[c_rnd[1], end_pt_cav[1]], z=[c_rnd[2], end_pt_cav[2]],
                mode='lines', line=dict(color='lime', width=6),
                name="Filter CAV"
            ))
            fig.add_trace(go.Cone(
                x=[end_pt_cav[0]], y=[end_pt_cav[1]], z=[end_pt_cav[2]],
                u=[v_norm_cav[0]], v=[v_norm_cav[1]], w=[v_norm_cav[2]],
                sizemode="absolute", sizeref=scale_cav*0.15, anchor="tail",
                colorscale=[[0, 'lime'], [1, 'lime']], showscale=False,
                name="CAV Head"
            ))

    fig.update_layout(
        title=f"Latent Space PCA - Layer {layer_id}<br><sup>Variance Explained: PC1: {var_explained[0]:.1f}%, PC2: {var_explained[1]:.1f}%, PC3: {var_explained[2]:.1f}%</sup>",
        scene=dict(xaxis_title="PC 1", yaxis_title="PC 2", zaxis_title="PC 3"),
        width=900, height=700, legend=dict(itemsizing='constant', x=0.85, y=0.9)
    )
    fig.show()

# ==========================================
# 4. Execute
# ==========================================
for l in LAYERS:
    plot_layer(l)

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd

# ==========================================
# 2D Density Visualization Function
# ==========================================
def plot_layer_2d_density(layer_id):
    print(f"Loading data for Layer {layer_id}...")
    X, y = load_layer_data(layer_id, DATA_DIR, ANTI_CONCEPT_DIR, NUM_RANDOM_RUNS, MAX_RANDOM_SAMPLES)
    if X is None: return

    pca = PCA(n_components=2)
    X_pca = pca.fit_transform(X)
    
    df = pd.DataFrame({'PC1': X_pca[:, 0], 'PC2': X_pca[:, 1], 'Label': y})
    plt.figure(figsize=(12, 8))
    
    palette = {
        'Random (Null)': 'grey', 'closed screen': 'orange',
        'open screen': 'blue', 'Alpha_O2_low': 'red', 'Alpha_O2_high': 'lime'
    }
    
    hue_order = ['Random (Null)', 'closed screen', 'open screen', 'Alpha_O2_low', 'Alpha_O2_high']
    present_labels = [l for l in hue_order if l in df['Label'].unique()]
    
    # Draw contours
    sns.kdeplot(
        data=df, x='PC1', y='PC2', hue='Label', hue_order=present_labels,
        palette=palette, fill=True, alpha=0.4, levels=6, linewidths=1.5, warn_singular=False
    )
    
    # Scatter concepts
    for lbl, col in [('Alpha_O2_low', 'red'), ('Alpha_O2_high', 'lime')]:
        sub_df = df[df['Label'] == lbl]
        if not sub_df.empty:
            sns.scatterplot(
                data=sub_df, x='PC1', y='PC2', color=col, edgecolor='black', 
                marker='D', s=40, zorder=6, label=lbl
            )

    # --- 1. TARGET GRADIENT ARROW (Black) ---
    grad_data = load_pkl(os.path.join(GRAD_DIR, f"target_gradients_layer_{layer_id}.pkl"))
    if grad_data is not None:
        mean_grad = np.mean(grad_data, axis=0) 
        proj_grad = np.dot(mean_grad, pca.components_.T)
        
        idx_neg = np.where(y == 'closed screen')[0]
        idx_pos = np.where(y == 'open screen')[0]
        
        if len(idx_neg) > 0 and len(idx_pos) > 0:
            c_neg = X_pca[idx_neg].mean(axis=0)
            c_pos = X_pca[idx_pos].mean(axis=0)
            
            v_norm = proj_grad / np.linalg.norm(proj_grad)
            scale = np.linalg.norm(c_pos - c_neg)
            end_pt = c_neg + (v_norm * scale)
            
            plt.annotate(
                '', xy=end_pt, xytext=c_neg,
                arrowprops=dict(facecolor='black', edgecolor='black', shrink=0, width=2, headwidth=10),
                zorder=10 
            )
            plt.text(end_pt[0], end_pt[1], " Gradient", color='black', fontsize=12, fontweight='bold', zorder=10)

    # --- 2. FILTER CAV ARROW (Green) ---
    cav_data = load_pkl(os.path.join(CAV_DIR, f"cavs_concept_set_layer_{layer_id}.pkl"))
    if cav_data is not None and 'filter' in cav_data:
        mean_cav = np.mean(cav_data['filter'], axis=0)
        proj_cav = np.dot(mean_cav, pca.components_.T)
        
        idx_rnd = np.where(y == 'Random (Null)')[0]
        idx_concept = np.where(y == 'Alpha_O2_high')[0]
        
        if len(idx_rnd) > 0 and len(idx_concept) > 0:
            c_rnd = X_pca[idx_rnd].mean(axis=0)
            c_concept = X_pca[idx_concept].mean(axis=0)
            
            v_norm_cav = proj_cav / np.linalg.norm(proj_cav)
            scale_cav = np.linalg.norm(c_concept - c_rnd)
            end_pt_cav = c_rnd + (v_norm_cav * scale_cav)
            
            plt.annotate(
                '', xy=end_pt_cav, xytext=c_rnd,
                arrowprops=dict(facecolor='lime', edgecolor='black', shrink=0, width=2, headwidth=10),
                zorder=10
            )
            plt.text(end_pt_cav[0], end_pt_cav[1], " CAV", color='green', fontsize=12, fontweight='bold', zorder=10)

    # Labels and formatting
    var_explained = pca.explained_variance_ratio_ * 100
    plt.title(f"2D Latent Space Density - Layer {layer_id}", fontsize=16)
    plt.xlabel(f"Principal Component 1 ({var_explained[0]:.1f}% Variance)", fontsize=12)
    plt.ylabel(f"Principal Component 2 ({var_explained[1]:.1f}% Variance)", fontsize=12)
    
    plt.legend(bbox_to_anchor=(1.05, 1), loc='upper left', borderaxespad=0.)
    plt.tight_layout()
    plt.show()

# ==========================================
# Execute
# ==========================================
for l in LAYERS:
    plot_layer_2d_density(l)